### Project 1 - Alternative Solution

In this project our goal is still to validate one nested dictionary against a template dictionary.

The rules are the same:

- every key in the template is required,
- extra keys are not allowed,
- nested dictionaries may appear at any depth,
- and leaf values must have the type specified by the template.

This time, though, we're going to solve the problem from a different angle.

Instead of recursively comparing the **data dictionary** and the **template dictionary** to each other, we're going to first convert each one into a simpler representation: a flat **path → type signature**.

Once both nested dictionaries have been reduced to that form, the actual validation becomes an ordinary comparison of two flat dictionaries.


For example, we'll use the same template:

In [1]:
template = {
    'user_id': int,
    'name': {
        'first': str,
        'last': str
    },
    'bio': {
        'dob': {
            'year': int,
            'month': int,
            'day': int
        },
        'birthplace': {
            'country': str,
            'city': str
        }
    }
}


And we'll use the same three pieces of data so that we can check our results against the original project requirements.

In [2]:
john = {
    'user_id': 100,
    'name': {
        'first': 'John',
        'last': 'Cleese'
    },
    'bio': {
        'dob': {
            'year': 1939,
            'month': 11,
            'day': 27
        },
        'birthplace': {
            'country': 'United Kingdom',
            'city': 'Weston-super-Mare'
        }
    }
}

eric = {
    'user_id': 101,
    'name': {
        'first': 'Eric',
        'last': 'Idle'
    },
    'bio': {
        'dob': {
            'year': 1943,
            'month': 3,
            'day': 29
        },
        'birthplace': {
            'country': 'United Kingdom'
        }
    }
}

michael = {
    'user_id': 102,
    'name': {
        'first': 'Michael',
        'last': 'Palin'
    },
    'bio': {
        'dob': {
            'year': 1943,
            'month': 'May',
            'day': 5
        },
        'birthplace': {
            'country': 'United Kingdom',
            'city': 'Sheffield'
        }
    }
}


The final function should still behave like this:

* `validate(john, template) --> (True, '')`
* `validate(eric, template) --> (False, 'mismatched keys: bio.birthplace.city')`
* `validate(michael, template) --> (False, 'bad type: bio.dob.month')`


##### Solution

There are a number of ways we could represent the structure of a nested dictionary.

One useful way is to think of every value as having an **address**.

For example:

- `user_id`
- `name.first`
- `name.last`
- `bio.dob.year`
- `bio.birthplace.city`

If we associate each one of those paths with a type, then a nested dictionary can be described by a completely flat dictionary.

So our plan is:

1. Write a small helper for constructing dotted paths.
2. Walk the template and build its expected path/type signature.
3. Walk the data and build its actual path/type signature.
4. Compare the two signatures.
5. Once that works, replace return codes with exceptions.


Let's start with the smallest piece: joining a parent path and a new key.

Keeping this in a helper avoids sprinkling special cases for the root path throughout the rest of the code.

In [3]:
def make_path(parent, key):
    key = str(key)
    if parent:
        return parent + '.' + key
    return key


print(make_path('', 'bio'))
print(make_path('bio.birthplace', 'city'))


bio
bio.birthplace.city


Now let's write a generator that walks a nested dictionary and yields one `(path, type)` pair at a time.

There is one wrinkle: the template stores **type objects** at its leaves (`int`, `str`, and so on), while the data stores actual values.

So we'll give the function a `template_mode` flag:

- in template mode, a leaf already *is* the expected type;
- in data mode, we use `type(value)` to discover the actual type.

We'll also include dictionary-valued nodes themselves in the signature. That is important because it lets us distinguish a missing nested key from a value that should have been a dictionary but wasn't.


In [4]:
def walk_signature(mapping, template_mode=False, path=''):
    for key, value in mapping.items():
        current_path = make_path(path, key)

        if isinstance(value, dict):
            yield current_path, dict
            yield from walk_signature(
                value,
                template_mode=template_mode,
                path=current_path
            )
        else:
            value_type = value if template_mode else type(value)
            yield current_path, value_type


Let's see what the template looks like after flattening it.

In [5]:
template_items = list(walk_signature(template, template_mode=True))

for path, expected_type in template_items:
    print(f'{path:<25} -> {expected_type.__name__}')


user_id                   -> int
name                      -> dict
name.first                -> str
name.last                 -> str
bio                       -> dict
bio.dob                   -> dict
bio.dob.year              -> int
bio.dob.month             -> int
bio.dob.day               -> int
bio.birthplace            -> dict
bio.birthplace.country    -> str
bio.birthplace.city       -> str


And now the valid data:

In [6]:
john_items = list(walk_signature(john))

for path, actual_type in john_items:
    print(f'{path:<25} -> {actual_type.__name__}')


user_id                   -> int
name                      -> dict
name.first                -> str
name.last                 -> str
bio                       -> dict
bio.dob                   -> dict
bio.dob.year              -> int
bio.dob.month             -> int
bio.dob.day               -> int
bio.birthplace            -> dict
bio.birthplace.country    -> str
bio.birthplace.city       -> str


The useful thing here is that the nested shape has disappeared from the validation problem.

At this point both sides can be turned into ordinary dictionaries:

```python
{
    'bio.dob.month': int,
    'bio.birthplace.city': str,
    ...
}
```

Let's write a helper for that.


In [7]:
def build_signature(mapping, template_mode=False):
    return dict(walk_signature(mapping, template_mode=template_mode))


expected = build_signature(template, template_mode=True)
actual = build_signature(john)

print(expected == actual)


True


For John, the signatures are identical, which is exactly what we want.

Before writing the whole validator, let's inspect the two failure cases.

First Eric, who is missing `bio.birthplace.city`.

In [8]:
expected = build_signature(template, template_mode=True)
eric_actual = build_signature(eric)

missing_paths = [path for path in expected if path not in eric_actual]
extra_paths = [path for path in eric_actual if path not in expected]

print('missing:', missing_paths)
print('extra:', extra_paths)


missing: ['bio.birthplace.city']
extra: []


Good. The missing path is now very easy to identify.

Now let's inspect Michael. His keys are all present, so the structure should match, but one path should have the wrong type.

In [9]:
michael_actual = build_signature(michael)

for path, expected_type in expected.items():
    if path in michael_actual and michael_actual[path] is not expected_type:
        print(
            path,
            'expected', expected_type.__name__,
            'found', michael_actual[path].__name__
        )


bio.dob.month expected int found str


So we now have all the pieces we need.

The validator itself does not need to recurse at all. The recursion has already happened while building the two signatures.

We'll validate in three passes:

1. Look for required paths that are missing.
2. For paths that exist, compare their types.
3. Look for extra paths that were not present in the template.

The order matters a little. Checking expected paths first means a missing required path is reported before unrelated extra data.


In [10]:
def validate(data, template):
    if not isinstance(data, dict):
        return False, 'bad type: <root>'

    if not isinstance(template, dict):
        raise TypeError('template must be a dictionary')

    expected = build_signature(template, template_mode=True)
    actual = build_signature(data)

    # Required paths and their types.
    for path, expected_type in expected.items():
        if path not in actual:
            return False, f'mismatched keys: {path}'

        if actual[path] is not expected_type:
            return False, f'bad type: {path}'

    # No extra paths are allowed.
    for path in actual:
        if path not in expected:
            return False, f'mismatched keys: {path}'

    return True, ''


Now let's test it against the three required examples.

In [11]:
persons = (
    (john, 'John'),
    (eric, 'Eric'),
    (michael, 'Michael')
)

for person, name in persons:
    is_ok, err_msg = validate(person, template)
    print(f'{name}: valid={is_ok}, error={err_msg!r}')


John: valid=True, error=''
Eric: valid=False, error='mismatched keys: bio.birthplace.city'
Michael: valid=False, error='bad type: bio.dob.month'


And let's make those expectations executable so that a later change cannot accidentally break the project requirements.

In [12]:
assert validate(john, template) == (True, '')
assert validate(eric, template) == (
    False,
    'mismatched keys: bio.birthplace.city'
)
assert validate(michael, template) == (
    False,
    'bad type: bio.dob.month'
)

print('Required project tests passed.')


Required project tests passed.


Let's try a few additional cases.

Because dictionary nodes themselves appear in the signature, the validator can also report a wrong container type at the correct path.

In [13]:
wrong_name = dict(john)
wrong_name['name'] = 'John Cleese'

extra_key = dict(john)
extra_key['active'] = True

print(validate(wrong_name, template))
print(validate(extra_key, template))


(False, 'bad type: name')
(False, 'mismatched keys: active')


One detail worth pointing out is the use of:

```python
actual[path] is expected_type
```

instead of:

```python
isinstance(value, expected_type)
```

For this project I want an `int` template to mean exactly `int`.

That avoids Python's slightly surprising behavior where `bool` is a subclass of `int`, so `isinstance(True, int)` is `True`.


In [14]:
bool_id = dict(john)
bool_id['user_id'] = True

print(validate(bool_id, template))
assert validate(bool_id, template) == (False, 'bad type: user_id')


(False, 'bad type: user_id')


So far we've returned a `(state, error)` tuple because that matches the project specification.

For application code, though, exceptions can make the successful path cleaner. Let's rework only the public validation layer while keeping the signature-building idea unchanged.

We'll use two specific exceptions so callers can distinguish structural problems from type problems.


In [15]:
class SchemaError(Exception):
    pass


class SchemaKeyMismatch(SchemaError):
    pass


class SchemaTypeMismatch(SchemaError, TypeError):
    pass


Now the exception-based validator can compare exactly the same signatures, but raise as soon as it finds the first problem.

In [16]:
def validate_or_raise(data, template):
    if not isinstance(data, dict):
        raise SchemaTypeMismatch('bad type: <root>')

    if not isinstance(template, dict):
        raise TypeError('template must be a dictionary')

    expected = build_signature(template, template_mode=True)
    actual = build_signature(data)

    for path, expected_type in expected.items():
        if path not in actual:
            raise SchemaKeyMismatch(f'mismatched keys: {path}')

        if actual[path] is not expected_type:
            raise SchemaTypeMismatch(f'bad type: {path}')

    for path in actual:
        if path not in expected:
            raise SchemaKeyMismatch(f'mismatched keys: {path}')


Then we can use it in the usual `try` / `except` style.

John produces no exception, while Eric and Michael can be handled differently.

In [17]:
validate_or_raise(john, template)
print('John is valid')

for person in (eric, michael):
    try:
        validate_or_raise(person, template)
    except SchemaKeyMismatch as ex:
        print('structure problem ->', ex)
    except SchemaTypeMismatch as ex:
        print('type problem      ->', ex)


John is valid
structure problem -> mismatched keys: bio.birthplace.city
type problem      -> bad type: bio.dob.month


There is one more useful improvement we can make.

If the same template is going to be used to validate many incoming records, rebuilding its signature every single time is unnecessary.

We can "compile" the template once and reuse the flat expected signature.


In [18]:
def compile_template(template):
    if not isinstance(template, dict):
        raise TypeError('template must be a dictionary')

    return build_signature(template, template_mode=True)


def validate_compiled(data, expected):
    if not isinstance(data, dict):
        return False, 'bad type: <root>'

    actual = build_signature(data)

    for path, expected_type in expected.items():
        if path not in actual:
            return False, f'mismatched keys: {path}'
        if actual[path] is not expected_type:
            return False, f'bad type: {path}'

    for path in actual:
        if path not in expected:
            return False, f'mismatched keys: {path}'

    return True, ''


Now the template work is done only once:

In [19]:
compiled_template = compile_template(template)

for person, name in persons:
    result = validate_compiled(person, compiled_template)
    print(name, '->', result)


John -> (True, '')
Eric -> (False, 'mismatched keys: bio.birthplace.city')
Michael -> (False, 'bad type: bio.dob.month')


This version reaches the same required answers, but the design is quite different from a validator that recursively compares the two dictionaries level-by-level.

Here recursion is used only to **describe** each nested dictionary.

After that, validation is performed against a flat representation, which has a few nice properties:

- paths are already available for error messages,
- structure and type checking become simple dictionary operations,
- the expected schema can be precomputed and reused,
- and the comparison logic is easy to test independently from tree traversal.

For a real production API we would normally reach for a dedicated schema library, but this is a useful exercise because it separates two ideas that often get mixed together: **walking nested data** and **validating a schema**.
